# Getting started with the ball-by-ball cricket dataset

Every ball of Tests, ODIs, T20Is and twelve T20 leagues, with Cricsheet's facts on each ball and, where Cricinfo tracked it, the line, length, shot and wagon wheel. This notebook loads the files, joins them, and makes four charts. The files join on `match_id`, `innings` and `delivery_number`, never on names.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Kaggle mounts a dataset under a path that differs between notebook versions, so look for the file
DATA = next(f.parent for root in (Path("/kaggle/input"), Path("data/output")) if root.exists() for f in root.rglob("ball_by_ball_data.csv"))
KEY = ["match_id", "innings", "delivery_number"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## 1. Load and join

The ball file holds Cricsheet's data. `cricinfo_tracking.csv` has one row per ball that Cricinfo tracked, and `cricinfo_text_predictions.csv` has predicted labels for balls that have commentary but no tracking. A left join keeps every ball and leaves the Cricinfo cells empty where there is nothing.

In [ ]:
balls = pd.read_csv(DATA / "ball_by_ball_data.csv",
                    usecols=KEY + ["start_date", "event", "striker", "bowler", "runs_off_bat", "extras", "wicket"])
tracking = pd.read_csv(DATA / "cricinfo_tracking.csv", low_memory=False)
predictions = pd.read_csv(DATA / "cricinfo_text_predictions.csv", low_memory=False)
balls["year"] = balls.start_date.str[:4].astype(int)
print(f"{len(balls):,} balls, {len(tracking):,} with Cricinfo tracking, {len(predictions):,} with a predicted label")

## 2. How much of the data has tracking

Cricsheet covers every ball from 2001. Cricinfo's line and length start around 2014, so the share filled depends on the era.

In [ ]:
tracked = balls.merge(tracking, on=KEY, how="left")
tracked["era"] = pd.cut(tracked.year, [0, 2013, 2019, 2100], labels=["2001 to 2013", "2014 to 2019", "2020 on"])
columns = {"ball_line": "line", "ball_length": "length", "shot_played": "shot", "wagon_x": "wagon point"}
share = tracked.groupby("era", observed=True)[list(columns)].agg(lambda s: s.notna().mean()).rename(columns=columns)
ax = share.plot.bar(figsize=(7, 4), rot=0)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
ax.set_ylabel("share of balls filled")
ax.set_title("Cricinfo tracking by era")
plt.show()

## 3. Line and length in the IPL since 2020

Where the ball pitched, and what it cost. Runs here are runs off the bat.

In [ ]:
ipl = tracked[(tracked.event == "Indian Premier League") & (tracked.year >= 2020)].dropna(subset=["ball_line", "ball_length"])
lengths = ["yorker", "full toss", "full", "good length", "short of a good length", "short"]
lines = ["down leg", "on the stumps", "outside offstump", "wide outside offstump"]
grid = pd.crosstab(ipl.ball_length, ipl.ball_line, normalize=True).reindex(index=lengths, columns=lines).fillna(0)

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.imshow(grid.values, cmap="Oranges")
ax.set_xticks(range(len(lines)), lines, rotation=20)
ax.set_yticks(range(len(lengths)), lengths)
for i in range(len(lengths)):
    for j in range(len(lines)):
        ax.text(j, i, f"{100 * grid.values[i, j]:.0f}%", ha="center", va="center", fontsize=9)
ax.set_title("Share of IPL balls by length and line, 2020 on")
plt.show()

ipl.groupby("ball_length").agg(balls=("runs_off_bat", "size"), runs_per_ball=("runs_off_bat", "mean"), wicket_rate=("wicket", "mean")).reindex(lengths).round(3)

## 4. A batter's wagon wheel

`wagon_angle` is 0 straight behind the batter and runs clockwise, mirrored for left-handers, so every batter reads the same way. `wagon_distance` is the fraction of the way to the boundary, and can pass 1.

In [ ]:
batter = "V Kohli"
shots = balls[(balls.striker == batter) & (balls.year >= 2022)].merge(tracking[KEY + ["wagon_angle", "wagon_distance"]], on=KEY).dropna(subset=["wagon_angle"])
colour = np.where(shots.runs_off_bat >= 4, "#eb6c36", "#4f5d75")

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.scatter(np.radians(shots.wagon_angle), shots.wagon_distance, c=colour, s=10, alpha=0.6)
ax.set_rmax(1.2)
ax.set_title(f"{batter} since 2022: {len(shots):,} tracked shots\n(orange: four or six)")
plt.show()

## 5. Using the predicted labels

Before 2014 Cricinfo recorded no length, but many commentary lines describe it. `cricinfo_text_predictions.csv` predicts a label for those balls with a confidence between 0 and 1. A prediction at 0.7 matched Cricinfo about 70% of the time on matches the model never saw, and `length_explicit` is 1 when the commentary names the length outright. The labels are predictions, so filter on confidence.

Here is how the IPL's first six seasons look by predicted length, keeping predictions at 0.6 or more.

In [ ]:
early = balls[(balls.event == "Indian Premier League") & (balls.year <= 2013)].merge(predictions[KEY + ["length", "length_confidence", "length_explicit"]], on=KEY)
sure = early[early.length_confidence >= 0.6]
print(f"{len(sure):,} of {len(early):,} early IPL balls with a predictioned length, {100 * sure.length_explicit.mean():.0f}% of them explicit")

summary = sure.groupby("length").agg(balls=("runs_off_bat", "size"), runs_per_ball=("runs_off_bat", "mean"), wicket_rate=("wicket", "mean")).reindex(lengths).round(3)
summary

## Notes

- An empty Cricinfo cell means Cricinfo had no value for that ball. About 4,200 matches have no Cricinfo data at all.
- Ball-by-ball and match data are from [Cricsheet](https://cricsheet.org/) under the Open Data Commons Attribution License 1.0. The tracking, the profile columns and the predicted labels come from Cricinfo or its commentary and are not covered by that licence, so credit Cricinfo and keep them to non-commercial use.
- The dataset page describes every column, and the project's documents explain the pairing and the predictions.